# Clase 2 — Bases de datos vectoriales

## De qué se trata

Los embeddings hacen posible comparar significado. Una base vectorial convierte esa comparación en un sistema que almacena, indexa, filtra, persiste y recupera evidencia para RAG.

## Referencia visual y conceptual

Inspirado en **AEM2L2 — Bases de datos vectoriales** y **AEM2L2.excalidraw**: registro vectorial, filtros, búsqueda exacta, ANN, HNSW e IVF.

## 1. Qué guarda un vector store

Cada chunk debe conservar el vector y todo lo necesario para interpretar el resultado.

| Campo | Ejemplo | Por qué importa |
|---|---|---|
| chunk_id | faq-023 | Conecta score, texto y fuente |
| embedding | [0.12, ...] | Permite búsqueda por similitud |
| content | política de vacaciones | Es la evidencia que verá el LLM |
| metadata | departamento, versión | Filtra y audita |
| configuración | modelo, métrica, fecha | Hace reproducible el índice |

In [ ]:
import matplotlib.pyplot as plt

layers = ["documento", "chunks", "embeddings", "índice", "Top-K"]
plt.figure(figsize=(9, 2.3))
plt.plot(range(len(layers)), [0]*len(layers), "o-", color="#1f4e79")
for i, label in enumerate(layers):
    plt.text(i, .1, label, ha="center")
plt.axis("off")
plt.title("Flujo de escritura y lectura de un vector store")
plt.show()

## 2. Búsqueda exacta y aproximada

La búsqueda exacta compara la consulta contra todos los vectores. ANN reduce candidatos para bajar latencia, pero puede omitir vecinos que el exacto recuperaría.

| Estrategia | Calidad | Latencia a gran escala | Uso en clase |
|---|---|---|---|
| k-NN exacto | Referencia máxima | Crece con el corpus | Baseline |
| HNSW | Alta con memoria | Baja | Grafo navegable |
| IVF | Ajustable con nprobe | Baja | Particiones candidatas |

In [ ]:
import matplotlib.pyplot as plt

corpus = [100, 1_000, 10_000, 100_000]
exact = [1, 10, 100, 1000]
ann = [1, 3, 8, 20]
plt.plot(corpus, exact, "o-", label="exacto")
plt.plot(corpus, ann, "s-", label="ANN")
plt.xscale("log")
plt.xlabel("cantidad de vectores")
plt.ylabel("trabajo relativo")
plt.title("Trade-off: precisión y velocidad")
plt.legend()
plt.grid(alpha=.25)
plt.show()

## 3. Normalización y sentido de los scores

Antes de indexar validá: dimensiones iguales, dtype compatible, vectores no nulos, métrica consistente y mismo modelo para corpus/consulta.

> Si el índice usa similitud o producto interno, **mayor score** suele ser mejor. Si usa distancia L2, **menor score** es mejor.

## Antes de ejecutar

Predicción: si ANN devuelve 4 de los 5 vecinos del baseline exacto, ¿cuál es Recall@5? ¿Qué otra métrica necesitás antes de aceptar el índice?

## Práctica guiada — ranking exacto y Recall@K

In [ ]:
import numpy as np

def top_k(scores, k=3):
    return np.argsort(scores)[::-1][:k].tolist()

scores = np.array([.34, .91, .72, .18, .66])
exact_top = top_k(scores, 3)
ann_top = [1, 2, 4]  # resultado aproximado ilustrativo
recall_at_3 = len(set(exact_top) & set(ann_top)) / len(exact_top)

print("exacto:", exact_top)
print("ANN:", ann_top)
print("Recall@3:", round(recall_at_3, 2))

## Tabla de benchmark que debe acompañar un experimento

| Backend | Modelo | Métrica | Top-K | Recall@K | p50 | p95 | Memoria |
|---|---|---|---:|---:|---:|---:|---:|
| Exacto | registrar | coseno/IP | 5 | 1.00 | medir | medir | medir |
| ANN | registrar | igual | 5 | medir | medir | medir | medir |

## Resultado esperado

Debés poder explicar el gráfico, relacionar el resultado con una decisión de ingeniería y modificar al menos un parámetro sin perder trazabilidad.

## Errores frecuentes

- Copiar valores o parámetros sin relacionarlos con el corpus y las consultas.
- Confundir una demo que funciona con una medición de calidad.
- Omitir IDs, fuentes, configuración o validaciones.

## Mini desafío

Aplicá la idea al FAQ de RR.HH. del proyecto integrador. Escribí una hipótesis, cambiá un parámetro y registrá qué métrica usarías para decidir si fue una mejora.

## Cierre

La próxima clase usa el vector store como retrieval y agrega contexto, prompt, generación y validación.

Después continuá con los notebooks E01–E10: allí la práctica va de un ejemplo mínimo a una implementación más robusta.